# Machine Learning Foundations — Module 1
## From everyday data to tensor operations

**Running example: Campus Study Coach.** We use **fictional students** and a deliberately hand-chosen score rule to understand how tensors implement data, parameters, transformations, predictions, and learning. The numbers do not measure real students and should never be used to evaluate them.

**Instructions:** Run the cells in order (CPU is enough). After each worked example, answer the short practice question before reading the next section.

In [1]:
# Standard setup. If PyTorch is missing in your notebook environment, run:
# %pip install torch
import torch
print('PyTorch version:', torch.__version__)
print('Device: CPU (no GPU required)')

PyTorch version: 2.10.0+cpu
Device: CPU (no GPU required)


## 1. A 1D tensor is one student's measurements
Ava studied **2 hours** and completed **3 practice sets**. Her measurements fit into one list: one axis, two entries. A weight vector also has one axis.

In [2]:
ava = torch.tensor([2.0, 3.0])
weights = torch.tensor([10.0, 5.0])
print('Ava:', ava.tolist(), 'shape:', tuple(ava.shape))
print('weights:', weights.tolist(), 'shape:', tuple(weights.shape))
print('weighted contribution:', (ava * weights).tolist())
print('dot product:', torch.dot(ava, weights).item())
assert tuple(ava.shape) == (2,)
assert torch.dot(ava,weights).item() == 35.0

Ava: [2.0, 3.0] shape: (2,)
weights: [10.0, 5.0] shape: (2,)
weighted contribution: [20.0, 15.0]
dot product: 35.0


**Try it:** Change Ava to 3 hours and 4 sets. What is the shape? What is the weighted contribution before adding the starting score?

## 2. A 2D tensor is the whole dataset
The dataset has **3 rows** (Ava, Ben, Cam) and **2 columns** (hours, practice sets). Both axes have a real-world meaning. Use indexing to ask who or what you want to select.

In [3]:
X = torch.tensor([[2.0,3.0], [4.0,1.0], [5.0,2.0]])
print('data:\n', X)
print('shape:', tuple(X.shape))
print('Ava:', X[0].tolist())
print('hours for everyone:', X[:,0].tolist())
assert tuple(X.shape) == (3,2)
assert X[:,0].tolist() == [2.0,4.0,5.0]

data:
 tensor([[2., 3.],
        [4., 1.],
        [5., 2.]])
shape: (3, 2)
Ava: [2.0, 3.0]
hours for everyone: [2.0, 4.0, 5.0]


**Try it:** Create a fourth fictional student with 1 hour and 4 practice sets using `torch.cat` or by reconstructing `X`. What is the new shape?

## 3. A parameterized function computes predictions
Our toy rule uses **10 points per study hour**, **5 points per practice set**, and **20 starting points**. The *same* parameter values operate on every student. This is a demonstration of computing predictions, not an actually trained score model.

In [4]:
w = torch.tensor([10.0, 5.0])
b = torch.tensor(20.0)
predictions = X @ w + b
print('dataset:', tuple(X.shape), 'weights:', tuple(w.shape), 'bias:', tuple(b.shape))
print('predicted scores:', predictions.tolist(), 'shape:', tuple(predictions.shape))
assert predictions.tolist() == [55.,65.,80.]
assert tuple(predictions.shape) == (3,)

dataset: (3, 2) weights: (2,) bias: ()
predicted scores: [55.0, 65.0, 80.0] shape: (3,)


**Try it:** Without running code, predict the score for 1 hour and 4 practice sets. Then verify with PyTorch. Why must the weight vector have 2 entries for this dataset?

## 4. Tensor transformations and broadcasting
Convert only the hours column from hours to minutes. The *values* change; the **shape stays (3,2)**. The model would need correspondingly adjusted weights (or retraining) to interpret minutes correctly. A scalar starting score broadcasts across every prediction.

In [5]:
minutes_X = X.clone()
minutes_X[:,0] = minutes_X[:,0] * 60
print('time in minutes:\n', minutes_X)
print('shape unchanged:', tuple(minutes_X.shape))
raw_scores = X @ w
print('before bias:', raw_scores.tolist())
print('after broadcasting scalar bias:', (raw_scores + 20).tolist())
assert tuple(minutes_X.shape)==(3,2)

time in minutes:
 tensor([[120.,   3.],
        [240.,   1.],
        [300.,   2.]])
shape unchanged: (3, 2)
before bias: [35.0, 45.0, 60.0]
after broadcasting scalar bias: [55.0, 65.0, 80.0]


**Try it:** If you switch to minutes, what weight should replace **10 points per hour** so the predictions stay unchanged? Verify numerically.

## 5. Repeated measurements introduce a 3D tensor
Suppose we record **2 students × 3 days × 2 measurements**. The new day axis preserves the temporal organization. Averaging across `dim=1` means average across the 3 days, retaining the student and feature axes.

In [6]:
logs = torch.tensor([[[1.,2.],[2.,3.],[3.,4.]],
                     [[2.,0.],[4.,2.],[6.,4.]]])
weekly_avg = logs.mean(dim=1)
print('daily logs shape:', tuple(logs.shape))
print('weekly averages:\n', weekly_avg)
print('weekly averages shape:',tuple(weekly_avg.shape))
assert tuple(logs.shape)==(2,3,2)
assert weekly_avg.tolist()==[[2.,3.],[4.,2.]]

daily logs shape: (2, 3, 2)
weekly averages:
 tensor([[2., 3.],
        [4., 2.]])
weekly averages shape: (2, 2)


**Try it:** Use `.sum(dim=1)` instead of `.mean(dim=1)`. What changes in the values? Does the shape change?

## 6. Pictures use a 4D tensor
A toy grayscale picture has **1 channel, 3 pixel rows, and 3 pixel columns**. Two students' pictures form a batch with shape **(2,1,3,3)**. Flattening *each image* creates a 9-element feature vector without throwing away pixels, but the original spatial layout is no longer explicit.

In [7]:
images = torch.tensor([[[[0.,1.,0.],[1.,1.,1.],[0.,1.,0.]]],
                       [[[1.,0.,1.],[0.,1.,0.],[1.,0.,1.]]]])
flat = images.flatten(start_dim=1)
print('images shape:',tuple(images.shape))
print('flattened shape:',tuple(flat.shape))
print('Ava flattened:',flat[0].tolist())
assert tuple(images.shape)==(2,1,3,3)
assert tuple(flat.shape)==(2,9)

images shape: (2, 1, 3, 3)
flattened shape: (2, 9)
Ava flattened: [0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0]


**Try it:** What shape represents 4 students, each with a 3-channel 32×32 color picture, using (student, channel, height, width) order?

## 7. 5D is one more meaningful axis
When each student uploads two picture snapshots, add a **day** axis between student and channel. More axes do not imply extra mathematical mystery: they encode more independent questions about the data.

In [8]:
snapshots = torch.stack([images, images*0.5],dim=1)
print('snapshots:',tuple(snapshots.shape))
print('axes: student, day, channel, height, width')
assert tuple(snapshots.shape)==(2,2,1,3,3)

snapshots: (2, 2, 1, 3, 3)
axes: student, day, channel, height, width


## 8. Learning also consists of tensor operations
We invent fictional desired outputs **[60,70,75]** and compare them with current model predictions. Mean squared error is the mean of the squared differences. PyTorch autograd calculates gradients of this loss with respect to our parameters.

In [9]:
target = torch.tensor([60.,70.,75.])
w_train = torch.tensor([10.,5.],requires_grad=True)
b_train = torch.tensor(20.,requires_grad=True)
prediction = X @ w_train + b_train
loss = ((prediction-target)**2).mean()
loss.backward()
print('current prediction:',prediction.detach().tolist())
print('initial loss:',round(loss.detach().item(),4))
print('weight gradient:',[round(v,4) for v in w_train.grad.tolist()])
print('bias gradient:',round(b_train.grad.item(),4))
assert loss.detach().item() == 25.
with torch.no_grad():
    w_train -= 0.01*w_train.grad
    b_train -= 0.01*b_train.grad
updated_loss = (((X @ w_train+b_train)-target)**2).mean()
print('loss after one gradient step:',round(updated_loss.detach().item(),4))
assert updated_loss.detach().item() < loss.detach().item()

current prediction: [55.0, 65.0, 80.0]
initial loss: 25.0
weight gradient: [-3.3333, -6.6667]
bias gradient: -3.3333
loss after one gradient step: 24.4185


**Try it:** Change the learning rate from 0.01 to 0.001 and run the training cell from the top. Compare the new loss. Explain *why* we need a loss function before we can train parameters.

## 9. Shape debugger
Before running an operation, name the axes and check compatibility.

In [10]:
bad_weights = torch.tensor([10.,5.,1.])
try:
    X @ bad_weights
except RuntimeError as exc:
    print('Expected shape mismatch:', str(exc).split('\n')[0])
print('Correct weight shape for X:',tuple(w.shape))


Expected shape mismatch: size mismatch, got input (3), mat (3x2), vec (3)
Correct weight shape for X: (2,)


## Instructor / self-check answer key (ungraded)
- One-student vector always has shape (2,) when it contains two features. For [3,4], weighted contribution is 3×10 + 4×5 = 50 before the starting bias.
- Four students × two features gives (4,2).
- With the 20-point bias, 1 hour and 4 sets gives 50.
- To preserve predictions when hours become minutes, use weight `10/60` points per minute.
- Summing daily logs, rather than averaging, yields [[6,9],[12,6]], still shape (2,2).
- Four RGB 32×32 images in (N,C,H,W) format yield shape (4,3,32,32).
- A loss is a numerical criterion that gradients can optimize; changing learning rate changes the size of the parameter update.

**Course design note:** The running example is instructional. It is not a recommendation to predict grades or assess real students from study habits.